# U2-L3：IMU 与里程计

**课程目标**

1. 能说明加速度计与陀螺仪各测什么，以及姿态角为什么要把两者的读数合起来算。
2. 能说明里程计的位置是怎么估出来的，走久了为什么会对不上实际位置。
3. 能使用 `get_imu()`、`get_odom()`、`reset_odom()` 读取姿态角与位置，并按固定间隔连续采样。
4. 能完成一次直线行走实验，比较指令距离、里程计读数与卷尺实测三个数，描述漂移现象。
5. 能把目标在里程坐标系里的位置，换算成「在我的前方多远、左侧多远」。


## 课程大纲映射（审核用）

| 轻任务 | 对应 cell | 核心概念 | 轻量概念 |
| --- | --- | --- | --- |
| ③ 姿态实验：手持缓慢倾斜看 RPY 变化 | 实验一 4.2、4.3 | — | C09 定时器（按固定间隔采样） |
| ④ 里程实验：直线走 2m，对比指令距离 vs 里程计读数（观察漂移） | 实验二 5.2–5.6 | — | — |
| 坐标变换：目标位置从里程系换到机器人自身系 | 三、实验二 5.1 | **A11 坐标变换**（odom 坐标系→机器人坐标系） | — |

- 编程线接口覆盖：`get_imu()`、`get_odom()` → `pose_2d`、`reset_odom()`。
- 姿态角取自 `odom.rpy`（融合结果）；`get_imu()` 读的是加速度计与陀螺仪的原始量。

> 本映射块用于大纲对齐审核；导出 PDF 版本时移除此块。


## 一、IMU 与姿态角

上一课从 IMU 里读出了加速度和角速度。这两个数各自来自一个元件，合在一起才回答得了「机器人现在是歪的还是正的」。

### 1.1 加速度计与陀螺仪

| 元件 | 量到的数 | 单位 | 单独拿它算姿态会怎样 |
| --- | --- | --- | --- |
| 加速度计 | 三个方向上的加速度；静止时量到的是重力 | m/s² | 静止时准，一动起来就把运动加速度混进来，算出的角度来回跳 |
| 陀螺仪 | 绕三个方向转动的快慢 | rad/s | 转动时准，但角度要靠累加得出，一点点误差会越积越多 |

手机横屏竖屏自动切换，用的就是这两个元件：静止时靠重力判断哪边朝下，转动过程中靠角速度跟上变化。

### 1.2 姿态角是融合出来的

两个元件各有各的毛病：一个怕动，一个怕久。把它们的读数放在一起互相补，再加上腿部关节的角度一起算，得到的姿态既跟得上变化，又不会越走越偏。这一步叫**姿态融合**。

K1 把融合后的姿态角放在里程计里输出，所以读姿态角用 `get_odom()`；`get_imu()` 拿到的是融合之前的原始量。

### 1.3 RPY 三个角

| 姿态角 | 绕的轴 | 对应的动作 |
| --- | --- | --- |
| roll 横滚 | 前后方向的轴 | 身体向左侧倒 / 向右侧倒 |
| pitch 俯仰 | 左右方向的轴 | 上身前倾 / 后仰 |
| yaw 航向 | 上下方向的轴 | 原地左转 / 右转 |

SDK 给出的单位是弧度，乘 180°/π 换成度比较好读。`odom.rpy` 里三个角按 roll、pitch、yaw 的顺序排列。


## 二、里程计与漂移

### 2.1 编码器与累加

机器人身上没有一个传感器能直接报出「我走了 1.3 米」。位置是算出来的：

- 每个关节电机里装了**编码器**，记录电机转过了多少角度。
- 腿每迈出一步，系统按步长把电机转角折算成这一小段的位移。
- 把每一小段位移一帧一帧加起来，就得到「相对起点走了多远」。

这种靠累加求总量的办法叫**积分**。`position` 里的 x、y、z 就是这么来的，单位是米。

### 2.2 漂移

折算步长时用的是理论值，实际情况不会完全照着理论走：脚落地时打滑、地面有点坡、转身时外侧腿多走了几厘米，每帧差一点点，走几百帧就差出去一截。

这种偏差每一次都很小、却只会越积越大的现象，叫**漂移**。足式机器人靠腿走路，脚跟地面之间没有固定的滚动关系，漂移比轮式机器人明显得多。

### 2.3 把当前位置设为原点

`reset_odom()` 把里程计的计数清零，从这一刻起重新算。做距离实验前调一次，读到的 x 就是这次走了多远，不用再做减法。


## 三、两个坐标系

机器人在同一时刻可以用两套坐标描述自己的位置，混用就会出错。

| 坐标系 | 原点在哪 | x 轴朝哪 | 用在哪 |
| --- | --- | --- | --- |
| 里程坐标系（odom） | 机器人出发的那一点 | 出发时机器人面朝的方向，之后不再变 | 里程计报出的 x、y |
| 机器人坐标系 | 机器人自己身上，跟着它走 | 机器人**当前**面朝的方向 | 速度指令 `set_velocity(vx, vy, vyaw)` |

两个坐标系的 x 轴只在出发那一刻重合。机器人转过身之后，`set_velocity(0.5, 0, 0)` 仍然是「朝我脸朝的方向走 0.5 m/s」，跟里程坐标系的 x 方向已经没什么关系了。

要把目标在里程坐标系里的位置，换成「在我的前方多远、左侧多远」，做一次换算：

```
dx = 目标x − 自己x
dy = 目标y − 自己y
前方 = dx × cos(yaw) + dy × sin(yaw)
左侧 = −dx × sin(yaw) + dy × cos(yaw)
```

`左侧` 算出来是负数，说明目标在右边。机器人要按位置去追一个目标时，这个换算每次都要用。


## 四、实验一：姿态观测

**实验名称**：姿态观测

**实验目标**：读出 K1 静止时的加速度与姿态角；按固定间隔连续采样，观察缓慢倾斜机身时 RPY 的变化。

⚠️ **安全**：本课有两次实验。

- **本次姿态观测**只读传感器数据，不发送运动指令，机器人保持坐姿或趴姿。
- **下一次里程实验**需要机器人行走。行走的安全要求：机器人必须处于已站立的状态才能进入 walk 模式，流程是 `prepare` → 人工拉起站立 → `walk`；停止时先切回 `prepare` 站稳，**严禁从 walk 直接切 damping**（站立状态下会瘫倒）。行走场地需留出 3 米以上直线空间，全程在教师监督下进行。

**前置准备**：K1 已开机并与电脑建立连接（见 U1-L1）；已完成 U1-L2（模式切换与安全）与 U2-L2（传感器速览）。

### 4.1 连接机器人


In [ ]:
from boosteros.robots.booster import BoosterRobot

robot = BoosterRobot()
print("已连接 K1")

### 4.2 静止时的读数

机器人安静地坐着或趴着，先读一组数当基准。


In [ ]:
import math

imu = robot.get_imu()
odom = robot.get_odom()

roll, pitch, yaw = odom.rpy
ax, ay, az = imu.linear_acceleration
wx, wy, wz = imu.angular_velocity

print(f"线加速度 (m/s²)：x={ax:6.2f}  y={ay:6.2f}  z={az:6.2f}")
print(f"角速度   (rad/s)：x={wx:6.3f}  y={wy:6.3f}  z={wz:6.3f}")
print()
print(f"roll  横滚 {math.degrees(roll):7.2f}°")
print(f"pitch 俯仰 {math.degrees(pitch):7.2f}°")
print(f"yaw   航向 {math.degrees(yaw):7.2f}°")

静止时，加速度里有一个轴的读数接近 9.8，那是重力压在这个轴上。角速度三个方向都接近 0，机器人没有在转。

姿态角三个数都不会正好是 0，正负几度属于正常：机器人是蹲坐着的，躯干本身就有倾角。记下这一组数，等会儿做对比。


### 4.3 按固定间隔连续采样

想看姿态变化的过程，就得一遍一遍地读。下面的循环每 0.5 秒采一次，连采 10 次，一共 5 秒。这种「每隔一段固定时间自动跑一遍采集」的做法，就是 ROS2 里的定时器在做的事。

运行这一格的同时，两名同学配合做动作：

1. 机器人保持坐姿或趴姿，一人扶稳躯干两侧；
2. 另一人缓慢把机身向左侧倾斜，大约 15 度，停住 1 秒；
3. 再缓慢扶正，最后向右侧倾斜一次，再扶正。

<font color="red">倾斜要慢，幅度不超过 20 度，全程扶稳，不要让机器人倾倒或跌落。</font>


In [ ]:
import time

print("开始采样，共 5 秒。请按提示缓慢倾斜机器人。")
print()
print(" 序号    acc_x   acc_y   acc_z     roll    pitch      yaw")
print("-" * 58)

for i in range(10):
    imu = robot.get_imu()
    odom = robot.get_odom()
    r, p, y = odom.rpy
    ax, ay, az = imu.linear_acceleration
    print(f"  {i:>2}   {ax:6.2f}  {ay:6.2f}  {az:6.2f} "
          f"{math.degrees(r):8.2f} {math.degrees(p):8.2f} {math.degrees(y):8.2f}")
    time.sleep(0.5)

把这 10 行和刚才的基准值对着看：

- 机身向一侧倒时，`roll` 跟着变大，向另一侧倒时符号反过来。哪个方向对应正值，看你们这组测出来的数。
- 机身前倾或后仰时，动的是 `pitch`。
- 倾斜过程中，加速度那个接近 9.8 的轴读数变小，重力被分摊到了另外的轴上。
- 晃动的时候 `yaw` 也会动，那是手在带：真正的原地转身要机器人自己在 walk 模式下做。

回到原位后，三个角会回到基准值附近。偏差一直回不来，说明姿态估计在这段时间里已经积累了误差。


## 五、实验二：里程实验

**实验名称**：直线行走与漂移观察

**实验目标**：把里程计清零，让机器人以 0.5 m/s 直线走 4 秒，比较指令距离、里程计读数与卷尺实测三个数，观察漂移。

<font color="red">本实验涉及行走。机器人须处于已站立状态才能进入 walk 模式（prepare → 人工拉起 → walk）；停止时先切 prepare 站稳，严禁从 walk 直接切 damping。场地留出 3 米以上直线空间，并在地面贴一条起点线，备好卷尺。</font>

**前置准备**：已完成本次实验的姿态观测部分；地面平整、无杂物、无障碍。

### 5.1 目标方位的换算

机器人还站在起点，先拿一个假设的目标点练一次坐标换算。改一改 `tx`、`ty` 的值，看算出来的前方、左侧距离怎么变。


In [ ]:
tx, ty = 2.0, 0.5          # 目标在里程坐标系里的位置，可以改

odom = robot.get_odom()
x, y, yaw = odom.pose_2d

dx, dy = tx - x, ty - y
forward = dx * math.cos(yaw) + dy * math.sin(yaw)
left = -dx * math.sin(yaw) + dy * math.cos(yaw)

print(f"机器人当前位姿：x={x:.2f}  y={y:.2f}  yaw={math.degrees(yaw):.1f}°")
print(f"目标位置      ：x={tx:.2f}  y={ty:.2f}")
print(f"换算到机器人身上：前方 {forward:.2f} m，左侧 {left:.2f} m")

`左侧` 是负数，说明目标在右边。机器人还没动，`yaw` 也接近 0，所以前方、左侧和目标点的 x、y 差不了多少。等机器人转身走一段之后再算一次，两组数的差别就出来了。

### 5.2 里程计清零

出发前把计数归零，这一趟走完读到的 x 就是本次的行走距离。


In [ ]:
robot.reset_odom()
print("清零后的位姿 [x, y, yaw]：", robot.get_odom().pose_2d.round(3))

### 5.3 进入行走模式

先从当前状态切到 `prepare`。这一格跑完之后，机器人处于等待拉起的站立就绪状态。


In [ ]:
robot.set_mode("prepare")
print("当前模式：", robot.get_mode())

<font color="red">停一下，确认机器人已被人工拉起到直立站立、周围 3 米内没有人和杂物、教师在场监督，再运行下一格。</font>


In [ ]:
robot.set_mode("walk")
print("当前模式：", robot.get_mode())

### 5.4 直线走 4 秒

速度指令发出去之后会一直生效，机器人持续前进；要停下必须再发一次速度为 0 的指令。


In [ ]:
robot.set_velocity(0.5, 0.0, 0.0)    # 以 0.5 m/s 向前
time.sleep(4.0)                       # 持续 4 秒
robot.set_velocity(0.0, 0.0, 0.0)    # 速度归零，停下

print("已停止。指令距离 = 0.5 m/s × 4 s = 2.00 m")

### 5.5 三个距离放在一起看

先读里程计，再用卷尺量出机器人实际走了多远，量的是起点线到此时脚底位置的距离。


In [ ]:
odom = robot.get_odom()
x, y, yaw = odom.pose_2d

print(f"里程计 x（前进方向）：{x:.3f} m")
print(f"里程计 y（横向偏移）：{y:.3f} m")
print(f"航向 yaw          ：{math.degrees(yaw):.1f}°")
print()
print("把卷尺量出的实距填进记录表，再算里程计与实距的差。")

三个数一般不会相等：

- **指令距离 2.00 m** —— 程序算出来的理论值，默认地面完全不打滑。
- **里程计读数** —— 机器人自己估的，编码器认为走了这么远。
- **卷尺实测** —— 真实值。

里程计和卷尺差出来的那部分，就是这一趟的漂移。指令是直线，可 `y` 也常常不是 0，那点横向偏移来自两腿迈步不对称、起步时机身轻微偏转这类因素，同样会累积。

### 5.6 停止收尾


In [ ]:
robot.set_mode("prepare")
print("当前模式：", robot.get_mode())

切回 `prepare` 后机器人原地站稳。之后按 U1-L2 学过的收尾流程处理：由人工扶着让机器人坐下，确认坐稳后再进入 `damping`。


## 六、观察与记录

| 观察项 | 你看到的 | 说明 |
| --- | --- | --- |
| 静止时的 roll / pitch / yaw |  | 记下基准值，单位度 |
| 静止时加速度哪个轴接近 9.8 |  | 这个轴大致朝着哪个方向？ |
| 向左侧倾斜时 roll 的读数 |  | 与基准值差多少？正的还是负的？ |
| 向右侧倾斜时 roll 的读数 |  | 和左侧那次符号相反吗？ |
| 倾斜时 acc_z 的变化 |  | 比 9.8 大还是小？ |
| 扶正后 roll 回到基准值了吗 |  | 差多少度？ |
| 指令距离 |  | 0.5 m/s × 4 s |
| 里程计 x |  | 机器人自己估的距离 |
| 卷尺实测距离 |  | 真实距离 |
| 里程计与实测之差 |  | 这一趟的漂移是多少？ |
| 里程计 y |  | 走直线时的横向偏移 |

**想一想**

1. 陀螺仪单独算角度时误差会越积越多，加速度计一动起来读数就不稳。姿态融合是怎么同时压住这两个问题的？
2. 实验二里 `y` 应该接近 0。你测到的 `y` 是多少？这几个厘米可能来自哪些环节？
3. 同一段行走程序连着跑三次，里程计读到的 x 会一模一样吗？为什么？


## 七、扩展阅读：定位靠什么兜底

里程计会漂，所以真正投入使用的室内机器人都会另外找一路定位信息来纠正它。展厅、仓库、工厂里常见的做法：

| 做法 | 在场地里加什么 | 机器人怎么用它 |
| --- | --- | --- |
| 地面标记 | 贴二维码、铺磁条、画引导线 | 相机或磁传感器读到标记，就知道自己在哪 |
| 无线基站 | 在墙角装 UWB 基站 | 测出与几个基站的距离，算出自己的位置 |
| 环境地图 | 事先用激光雷达扫一遍场地 | 运行时拿实时扫描和地图比对，反推位置 |

这些方案都要在场地里先做一遍部署和标定，机器人跑起来之后还要反复调。负责这项工作的岗位通常叫部署工程师或调试工程师，日常就是拿卷尺量误差、改参数、挪基站，直到机器人每天跑同一条路线都能停在同一个位置。

本课量出的那几个厘米漂移，到了现场就是他们要处理的活。
